# What are `cache()` and `persist()` in Spark?

One of the biggest advantages of Spark is **in-memory processing**, but by default Spark **does not keep the results of transformations in memory**.

Spark follows **Lazy Evaluation**, meaning it only executes transformations when an **action** (`count()`, `show()`, `collect()`, `write()`, etc.) is called.

If the same DataFrame is used multiple times, Spark **recomputes the entire execution plan every time**.

`cache()` and `persist()` solve this problem by storing the DataFrame after the first computation so it can be reused.

cache() is store dataframe temporarily(in memory) in computing environment.so it can be reused again.

Spark recomputes the entire lineage every time an action is executed. Using `cache()` or `persist()` stores the DataFrame in memory (or memory and disk), avoiding repeated computation and improving performance.

---

# Why Do We Need `cache()`?

Suppose we have the following pipeline:

```python
orders = (
    spark.read.parquet("/sales")
         .filter("amount > 1000")
         .join(customer_df,"customer_id")
         .groupBy("country")
         .sum("amount")
)
```

Now suppose we execute

```python
orders.show()

orders.count()

orders.write.parquet("/output")
```

### What Spark Does

```
Read Data
     ↓
Filter
     ↓
Join
     ↓
Aggregation
     ↓
Show()

----------------------------------

Read Data
     ↓
Filter
     ↓
Join
     ↓
Aggregation
     ↓
Count()

----------------------------------

Read Data
     ↓
Filter
     ↓
Join
     ↓
Aggregation
     ↓
Write()
```

Spark repeats the **same expensive transformations three times**.

---

# Using Cache

```python
orders.cache()

orders.count()     # Materializes the cache

orders.show()

orders.write.parquet("/output")
```

Now Spark executes

```
Read Data
     ↓
Filter
     ↓
Join
     ↓
Aggregation
     ↓
Cache

        ↓

Show()

Count()

Write()
```

The transformations are executed only **once**.

Every subsequent action reads the data directly from memory.

---

# What is `persist()`?

`persist()` is an advanced version of `cache()`.

It allows you to specify **where Spark should store the DataFrame**.

```python
from pyspark import StorageLevel

df.persist(StorageLevel.MEMORY_AND_DISK)
```

---

# Difference Between `cache()` and `persist()`

| cache() | persist() |
|-----------|------------|
| Easy to use | More flexible |
| Uses the default storage level | You choose the storage level |
| Best for most scenarios | Best for very large datasets |

---

# Storage Levels

| Storage Level | Description |
|---------------|-------------|
| MEMORY_ONLY | Store only in RAM |
| MEMORY_AND_DISK | Store in RAM, overflow to disk |
| DISK_ONLY | Store only on disk |
| MEMORY_ONLY_SER | Store serialized objects |
| OFF_HEAP | Store outside JVM heap |

---

# Real-Time Example 1: ETL Pipeline (Most Common)

Imagine an ETL pipeline:

```
Raw CSV

↓

Clean Data

↓

Remove Duplicates

↓

Join Customer Table

↓

Business Transformations

↓

Write to

• Delta Table

• Hive Table

• CSV Archive

• Validation Report
```

### Without Cache

Each write recomputes

```
CSV Read

↓

Cleaning

↓

Join

↓

Transformations

↓

Write Delta
```

Again

```
CSV Read

↓

Cleaning

↓

Join

↓

Transformations

↓

Write Hive
```

Again

```
CSV Read

↓

Cleaning

↓

Join

↓

Transformations

↓

Generate Report
```

Three expensive executions.

---

### With Cache

```python
customer_df = transform(raw_df)

customer_df.cache()

customer_df.count()

customer_df.write.format("delta").save(...)

customer_df.write.saveAsTable(...)

customer_df.write.csv(...)
```

Spark performs the transformation only once.

### Why It Helps

- Faster execution
- Reduced CPU usage
- Less disk I/O
- Lower cluster cost

---

# Real-Time Example 2: Dashboard Reporting

A dashboard needs

- Sales by Country
- Sales by Product
- Sales by Month
- Sales by Region

All use the same transformed DataFrame.

```python
sales = prepare_sales(raw_sales)

sales.cache()

sales.groupBy("country").sum("amount")

sales.groupBy("product").sum("amount")

sales.groupBy("month").sum("amount")

sales.groupBy("region").sum("amount")
```

### Without Cache

The entire ETL is executed four times.

### With Cache

Transformation runs once.

All reports use the cached data.

---

# Real-Time Example 3: Machine Learning

Feature engineering is expensive.

```python
features = create_features(df)

features.cache()

train_model(features)

evaluate(features)

predict(features)
```

Without cache

Feature engineering executes three times.

With cache

Features are created once.

---

# Real-Time Example 4: Data Quality Validation

Suppose your pipeline performs

```python
clean_df.count()

clean_df.describe()

clean_df.filter("salary IS NULL").count()

clean_df.groupBy("country").count()
```

Every action triggers the same transformations.

Better approach

```python
clean_df.cache()

clean_df.count()

clean_df.describe()

clean_df.groupBy("country").count()
```

---

# Real-Time Example 5: Large Dimension Table

Imagine

```
Customer Dimension

↓

Join Orders

↓

Join Payments

↓

Join Returns

↓

Join Shipments
```

Instead of reading Customer table four times

```python
customer.cache()

orders.join(customer)

payments.join(customer)

returns.join(customer)

shipment.join(customer)
```

Huge performance improvement.

---

# Real-Time Example 6: Delta MERGE

Suppose

```
Source Table

↓

Business Transformations

↓

MERGE INTO Delta

↓

Validation

↓

Audit Table
```

The transformed DataFrame is reused.

```python
final_df.cache()

merge_into_delta(final_df)

validate(final_df)

audit(final_df)
```

---

# Real-Time Example 7: Kafka Streaming Reference Data

Suppose

```
Kafka Stream

↓

Join Country Master

↓

Join Product Master
```

Country master changes only once per day.

```python
country_df.cache()

stream.join(country_df)
```

Instead of loading country data every micro-batch, Spark reuses the cached DataFrame.

---

# Real-Time Example 8: Exploratory Analysis in Databricks

A Data Scientist runs

```python
sales.show()

sales.describe()

sales.count()

sales.groupBy("country").count()

sales.filter("amount>1000")
```

Instead of recomputing every notebook cell

```python
sales.cache()

sales.count()
```

Subsequent operations become much faster.

---

# When Should You Use `persist()`?

Use `persist()` when:

- Dataset is larger than available memory.
- Memory-only caching may cause recomputation.
- You want Spark to spill cached partitions to disk.

Example

```python
from pyspark import StorageLevel

large_df.persist(StorageLevel.MEMORY_AND_DISK)
```

If RAM becomes full, Spark automatically stores the remaining partitions on disk.

---

# When Should You NOT Use Cache?

Do **not** cache when:

### DataFrame is used only once

```python
df.write.parquet(...)
```

No benefit.

---

### DataFrame is very small

Spark can recompute it quickly.

---

### Memory is limited

Caching unnecessary DataFrames can evict more useful cached data and reduce performance.

---

# Common Interview Questions

## Q1. Is `cache()` an action?

No.

It is a transformation hint.

Nothing is cached until an action executes.

```python
df.cache()

# No cache yet

df.count()

# Now cached
```

---

## Q2. Why do we call `count()` after `cache()`?

Because Spark is lazy.

`count()` forces Spark to compute the DataFrame and populate the cache.

---

## Q3. Which is better: `cache()` or `persist()`?

- Use `cache()` for most workloads where the dataset fits in memory.
- Use `persist()` when you need control over storage levels or when the dataset may exceed memory.

---

## Q4. What happens if memory is full?

### `MEMORY_ONLY`

Partitions that don't fit are not cached and are recomputed when accessed again.

### `MEMORY_AND_DISK`

Overflow partitions are written to disk and reused later without recomputation.

---

# Production Best Practices

- ✅ Cache only DataFrames reused multiple times.
- ✅ Cache after expensive transformations such as joins, aggregations, and filters.
- ✅ Trigger an action (for example, `count()`) immediately after caching to materialize it.
- ✅ Call `unpersist()` when the cached DataFrame is no longer needed.
- ✅ Use `persist(StorageLevel.MEMORY_AND_DISK)` for large production datasets.
- ✅ Monitor the **Storage** tab in the Spark UI to verify cache usage and memory consumption.

---

# Interview Summary

| Question | Answer |
|----------|---------|
| What is `cache()`? | Stores a DataFrame in memory for reuse, avoiding recomputation. |
| What is `persist()`? | Stores a DataFrame using a specified storage level (memory, disk, or both). |
| Why use them? | To improve performance when the same DataFrame is accessed multiple times. |
| When to use? | After expensive transformations that feed multiple downstream actions. |
| When not to use? | When the DataFrame is used only once or is inexpensive to recompute. |
| Most common production use? | ETL pipelines, reporting, machine learning, Delta Lake merges, data quality validation, repeated joins, and interactive analytics. |

### When to use cache():
 - DataFrame is reused multiple times (e.g., for reporting, validation, multiple writes)
- After expensive transformations (joins, aggregations, filters)
- Dataset fits comfortably in memory

df.cache()
df.count()  # Materializes the cache

### When to use persist():
- Dataset is too large to fit entirely in memory
- You want Spark to spill partitions to disk if memory is insufficient
- Need a specific storage strategy (e.g., DISK_ONLY, MEMORY_ONLY_SER)

from pyspark import StorageLevel


large_df.persist(StorageLevel.MEMORY_AND_DISK)

large_df.count()  # Materializes the persisted data

# Real-Time Use of `cache()` and `persist()` in Spark & PySpark

In Spark, every transformation is **lazy**. If the same DataFrame is used multiple times, Spark recomputes the entire lineage every time an action is executed. Using `cache()` or `persist()` stores the DataFrame in memory (or memory and disk), avoiding repeated computation and improving performance.

---

# Why Do We Need `cache()` and `persist()`?

Without caching:

```
Read CSV
      ↓
Filter
      ↓
Join
      ↓
Aggregation
      ↓
Action 1
```

If another action is performed:

```
Read CSV
      ↓
Filter
      ↓
Join
      ↓
Aggregation
      ↓
Action 2
```

Spark recomputes the entire pipeline for each action.

With caching:

```
Read CSV
      ↓
Filter
      ↓
Join
      ↓
Cache
      ↓
Action 1

      ↓

Action 2

      ↓

Action 3
```

Spark computes the DataFrame once and reuses it for subsequent actions.

---

# 1. Using `cache()`

## Use Case

Store the DataFrame in memory for repeated use.

### Example

```python
df = spark.read.csv("/sales")

sales_df = df.filter("amount > 1000")

sales_df.cache()

sales_df.count()

sales_df.show()

sales_df.write.mode("overwrite").parquet("/output")
```

### Real-Time Example

A transformed sales DataFrame is used for:

- Dashboard reporting
- Data validation
- Writing to Delta Lake

Instead of recomputing the transformation three times, cache it once.

---

# 2. Using `persist()`

`persist()` provides control over the storage level.

### Example

```python
from pyspark import StorageLevel

df.persist(StorageLevel.MEMORY_AND_DISK)
```

---

# Storage Levels

| Storage Level | Description |
|---------------|-------------|
| MEMORY_ONLY | Store only in memory |
| MEMORY_AND_DISK | Store in memory; spill to disk if needed |
| MEMORY_ONLY_SER | Store serialized objects in memory |
| DISK_ONLY | Store only on disk |
| OFF_HEAP | Store outside JVM heap (advanced use cases) |

---

# Difference Between `cache()` and `persist()`

| `cache()` | `persist()` |
|------------|-------------|
| Uses default storage level | Allows custom storage level |
| Equivalent to `MEMORY_AND_DISK` for DataFrames | Can use MEMORY_ONLY, DISK_ONLY, etc. |
| Simpler syntax | More flexible |

### Example

```python
df.cache()

# Equivalent to

df.persist(StorageLevel.MEMORY_AND_DISK)
```

---

# Real-Time Project Scenarios

## Scenario 1: Multiple Reports from the Same Dataset

```python
sales = (
    spark.read.parquet("/sales")
         .filter("amount > 1000")
)

sales.cache()

sales.groupBy("country").sum("amount")

sales.groupBy("product").sum("amount")

sales.groupBy("year").sum("amount")
```

### Why Cache?

The filtered dataset is reused for multiple reports, avoiding repeated reads and transformations.

---

## Scenario 2: Machine Learning

```python
features = feature_engineering(df)

features.cache()

train_model(features)

evaluate_model(features)

predict(features)
```

### Why Cache?

Feature engineering is computationally expensive. Cache the feature set and reuse it for training, evaluation, and prediction.

---

## Scenario 3: Data Quality Checks

```python
clean_df = transform(df)

clean_df.cache()

clean_df.count()

clean_df.filter("salary IS NULL").count()

clean_df.describe().show()
```

### Why Cache?

The same transformed data is used for multiple validation checks.

---

## Scenario 4: ETL Pipeline

```python
customer = clean_customer(raw_df)

customer.cache()

customer.write.parquet("/silver")

customer.write.jdbc(...)

customer.write.saveAsTable(...)
```

### Why Cache?

Avoid reprocessing the transformation before each write operation.

---

## Scenario 5: Large Dimension Table

```python
dim_customer = spark.read.parquet("/dimension/customer")

dim_customer.cache()

fact.join(dim_customer, "customer_id")
```

### Why Cache?

The same dimension table is joined with multiple fact tables.

---

## Scenario 6: Iterative Algorithms

```python
graph.cache()

for i in range(20):
    process(graph)
```

### Why Cache?

Algorithms like PageRank and graph processing repeatedly use the same dataset.

---

## Scenario 7: Interactive Data Analysis

```python
sales.cache()

sales.show()

sales.describe()

sales.groupBy("city").count()

sales.filter("amount > 1000")
```

### Why Cache?

Data analysts execute multiple queries against the same DataFrame during exploration.

---

## Scenario 8: Multiple Joins

```python
customer.cache()

orders.join(customer)

payments.join(customer)

returns.join(customer)
```

### Why Cache?

The customer DataFrame is reused across multiple joins.

---

## Scenario 9: Slowly Changing Dimension (SCD)

```python
existing_customer.cache()

new_customer.join(existing_customer)
```

### Why Cache?

The target dimension is referenced multiple times during merge operations.

---

## Scenario 10: Streaming Reference Data

```python
country_df.cache()

stream_df.join(country_df)
```

### Why Cache?

The static reference data is reused for every micro-batch.

---

# When to Use `persist()` Instead of `cache()`

Use `persist()` when:

- The dataset is too large to fit entirely in memory.
- You want Spark to spill partitions to disk if memory is insufficient.
- You need a specific storage strategy (e.g., `DISK_ONLY` for very large datasets).

### Example

```python
from pyspark import StorageLevel

large_df.persist(StorageLevel.MEMORY_AND_DISK)
```

---

# When NOT to Use `cache()`

Avoid caching when:

- The DataFrame is used only once.
- The DataFrame is very small (recomputing is inexpensive).
- Memory is limited and the cached data may evict more valuable datasets.
- The transformations are simple and inexpensive.

### Bad Example

```python
df = spark.read.csv("/employee")

df.cache()

df.write.parquet("/output")
```

The DataFrame is used only once, so caching provides no benefit.

---

# How to Remove Cache

```python
df.unpersist()
```

Clear all cached DataFrames.

```python
spark.catalog.clearCache()
```

---

# Interview Questions

### Q1. What is the difference between `cache()` and `persist()`?

| `cache()` | `persist()` |
|------------|-------------|
| Default storage level | User-defined storage level |
| Simple to use | Flexible |
| Best for most use cases | Best for large or specialized workloads |

---

### Q2. Is `cache()` an action?

No. `cache()` is a transformation hint. The DataFrame is not cached until an action such as `count()`, `show()`, or `collect()` is executed.

Example:

```python
df.cache()

# Nothing is cached yet

df.count()

# Data is now cached
```

---

### Q3. Why call `count()` after `cache()`?

Because Spark uses lazy evaluation. Calling an action materializes the cache so subsequent actions can reuse it.

---

### Q4. Which storage level is best for production?

`MEMORY_AND_DISK` is generally the safest choice because it allows Spark to spill partitions to disk if memory is insufficient, reducing the risk of recomputation.

---

### Q5. What happens if memory is full?

- With `MEMORY_ONLY`, partitions that don't fit are not cached and are recomputed when needed.
- With `MEMORY_AND_DISK`, Spark stores the overflow partitions on disk instead of recomputing them.

---

# Best Practices

- ✅ Cache or persist only DataFrames that are reused multiple times.
- ✅ Trigger an action (such as `count()`) after caching to materialize the cache.
- ✅ Call `unpersist()` after the DataFrame is no longer needed to free cluster memory.
- ✅ Prefer `persist(StorageLevel.MEMORY_AND_DISK)` for large production datasets.
- ✅ Monitor the Spark UI's **Storage** tab to verify that datasets are cached as expected.
- ✅ Avoid excessive caching, as it can lead to memory pressure and reduced overall cluster performance.

> **Interview Tip:** A common production pattern is to cache or persist an expensive DataFrame after joins and transformations when it is reused for multiple outputs—for example, writing to Delta Lake, generating reports, and performing data quality checks. This eliminates repeated computation and significantly improves job performance.